# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/widadfatimakhan/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane 4 — CTR / Engagement Opportunity Scoring.** One hand-written rule, on the same March 2026
slice my ML-04 contract defined. This is the baseline my Week-5 model has to beat.

The session's shape, followed in order: **check the belief → write the flag as a sentence → choose a
threshold and document it → rank by what is at stake → read the top of the list like a skeptic.**

| Card asks for | Lives in |
|---|---|
| Two signal checks, bucket table + n, one verdict word each (≥1 flag-linked) | §1 |
| One rule: a score, one reason code, an action label | §2 |
| Ranked queue written to `work/outputs/baseline_action_score.csv` | §2 |
| Top-of-list review: action, why, what would make it wrong | §3 |
| Weak picks + leakage check | §4 |

**Lane lock.** Confirming **Lane 4**, not switching. ML-04 measured what the queue is worth: the top
50 pages carry ~11,900 recoverable clicks a month, ~1.5% of the lane's observed clicks — enough to
be worth an editor's week.

## 0. Setup — the same slice ML-04 contracted

Nothing new is decided here. This rebuilds the exact page-month frame from the data contract:
March 2026, one row per page, positions `+1`-corrected (the zero-based fix from ML-04 §3.4).

In [1]:
%pip -q install duckdb

import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
print("token loaded:", bool(HF_TOKEN))

token loaded: True


In [2]:
import duckdb, pandas as pd, numpy as np
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 50)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL, MONTH = "hf://datasets/FlyRank/internship-warehouse", "2026-03"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

# Contract constants, carried forward unchanged from ML-04.
WINDOW_START, WINDOW_END = "2026-03-01", "2026-03-31"
DECISION_MOMENT = "2026-04-01"
MIN_IMPRESSIONS, MIN_ACTIVE_DAYS, MIN_POSITION = 500, 5, 1.0

pages = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions)                                                  AS impressions_31d,
           SUM(gsc_clicks)                                                       AS clicks_31d,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END)    AS days_with_impressions_31d,
           MAX(gsc_impressions)                                                  AS top_day_impressions,
           SUM(gsc_sum_position) FILTER (WHERE gsc_impressions > 0
                                           AND gsc_avg_position IS NOT NULL)     AS pos_num,
           SUM(gsc_impressions)  FILTER (WHERE gsc_impressions > 0
                                           AND gsc_avg_position IS NOT NULL)     AS pos_den,
           STDDEV_SAMP(gsc_avg_position) FILTER (WHERE gsc_impressions > 0
                                           AND gsc_avg_position IS NOT NULL)     AS position_volatility_31d,
           SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-18')  AS imp_last14,
           SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-04'
                                          AND report_date <  DATE '2026-03-18')  AS imp_prev14
    FROM {FACT}
    WHERE gsc_data_available IS TRUE          -- IS TRUE, never = TRUE
    GROUP BY 1, 2
""").df()

# +1: gsc_avg_position is zero-based (proved in ML-04 3.4), so 1-based = sum/impressions + 1
pages["avg_position_31d"] = pages["pos_num"] / pages["pos_den"].replace(0, np.nan) + 1
pages["ctr_pp"] = 100 * pages["clicks_31d"] / pages["impressions_31d"].replace(0, np.nan)
print(f"pages with GSC data in {MONTH}: {len(pages):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages with GSC data in 2026-03: 176,738


In [3]:
# Eligibility -> the lane slice the rule is allowed to act on
elig = ((pages.impressions_31d >= MIN_IMPRESSIONS) &
        (pages.days_with_impressions_31d >= MIN_ACTIVE_DAYS) &
        (pages.avg_position_31d >= MIN_POSITION))
lane = pages[elig].copy()

lane["log_impressions_31d"]      = np.log1p(lane.impressions_31d)
lane["top_day_impression_share"] = lane.top_day_impressions / lane.impressions_31d
lane["momentum_log14v14"]        = np.log((lane.imp_last14.fillna(0) + 1) /
                                          (lane.imp_prev14.fillna(0) + 1))

def position_tier(p):
    return ("top_3" if p <= 3 else "page_1" if p <= 10 else "striking" if p <= 20
            else "page_3_5" if p <= 50 else "deep")
lane["position_tier"] = lane.avg_position_31d.apply(position_tier)

print(f"eligible lane slice: {len(lane):,} pages, {lane.client_hash_id.nunique()} clients")
print(f"({len(pages) - len(lane):,} pages excluded: below the volume floor, too few active days, "
      f"or no usable position)")

eligible lane slice: 61,881 pages, 36 clients
(114,857 pages excluded: below the volume floor, too few active days, or no usable position)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### The decision contract (the session's four lines)

1. **The decision** — which pages get a **snippet review** this week?
2. **What "better" means** — CTR rises at a similar position, within six weeks of the rewrite.
3. **Who is eligible** — ≥500 impressions in March, ≥5 active days, a usable position. Not in
   cooldown (see the limitation below).
4. **Who confirms** — a person. The queue proposes; an editor decides.

### The flag, as one sentence

> **AMONG** eligible pages · **LOOK AT** click-through rate against pages at a similar position ·
> **IF** it is far enough below those peers to be worth an editor's time · **THEN** send it to
> snippet review, with the reason attached.

### Reason codes and action labels

| reason_code | action | meaning |
|---|---|---|
| `CTR_BELOW_POSITION_PEERS` | `SNIPPET_REVIEW` | measurably below peers at a similar position, and the shortfall is worth chasing |
| `WITHIN_PEER_RANGE` | `MONITOR` | not far enough below peers to act on — "no action yet" is a real answer |

**A cooldown belongs in line 3 and I cannot implement it.** The release has no record of which
pages were previously optimised, so I cannot exclude recently-touched pages. Stated as a known gap,
not silently skipped.

### Before the rule: check the two beliefs it rests on

The rule leans on two assumptions. Neither is checked by writing it more confidently, so each gets
a bucket table with **n** printed and one verdict word.

### Signal 1 (flag-linked) — does CTR really fall as position worsens?

This is the belief under FlyRank's CTR-fix logic and under my whole lane: comparing a page only to
pages at a similar position is fair *because* position drives CTR. If CTR were flat across
positions, the peer grouping would be decoration.

ML-04 raised a doubt here — the five documented tiers came out non-monotonic. So this test uses
**finer position bands** to see whether the coarse tiers were hiding the pattern.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

BANDS  = [0, 2, 3, 5, 10, 20, 50, np.inf]
LABELS = ["1.0-2", "2-3", "3-5", "5-10", "10-20", "20-50", "50+"]
lane["pos_band"] = pd.cut(lane.avg_position_31d, bins=BANDS, labels=LABELS)

s1 = lane.groupby("pos_band", observed=True).agg(
        n_pages=("ctr_pp", "size"), impressions=("impressions_31d", "sum"),
        clicks=("clicks_31d", "sum"))
s1["pooled_ctr_pp"] = (100 * s1.clicks / s1.impressions).round(3)
s1["share_zero_click"] = lane.groupby("pos_band", observed=True).clicks_31d.apply(lambda s: (s == 0).mean()).round(3)
print("SIGNAL 1 - pooled CTR by position band (n printed for every bucket)")
display(s1[["n_pages", "impressions", "pooled_ctr_pp", "share_zero_click"]])

FLOOR = 50   # signal-audit rule: no verdict from a bucket under ~50 rows
usable = s1[s1.n_pages >= FLOOR]
seq = usable.pooled_ctr_pp.tolist()
falls_all  = all(a >= b for a, b in zip(seq, seq[1:]))
falls_most = sum(a >= b for a, b in zip(seq, seq[1:])) >= len(seq) - 2
spread     = max(seq) / max(min(seq), 1e-9)

verdict_1 = ("CONFIRMED" if falls_all else
             "MIXED"     if falls_most or spread > 2 else
             "OPPOSITE"  if all(a <= b for a, b in zip(seq, seq[1:])) else "FALSE")
print(f"\nbuckets used for the verdict (n >= {FLOOR}): {len(usable)} of {len(s1)}")
print(f"best band {max(seq):.3f}% vs worst {min(seq):.3f}% -> {spread:.1f}x spread")
print(f"\nVERDICT 1: {verdict_1}")

SIGNAL 1 - pooled CTR by position band (n printed for every bucket)


,n_pages,impressions,pooled_ctr_pp,share_zero_click
pos_band,,,,
1.0-2,483,1991779.0,0.118,0.350
2-3,2378,9496453.0,0.358,0.133
3-5,11631,68052893.0,0.394,0.085
5-10,23572,99125620.0,0.303,0.134
10-20,11916,31744858.0,0.330,0.219
20-50,11152,56852702.0,0.145,0.270
50+,749,1619191.0,0.036,0.696



buckets used for the verdict (n >= 50): 7 of 7
best band 0.394% vs worst 0.036% -> 10.9x spread

VERDICT 1: MIXED


**Reading verdict 1 — and it is more specific than MIXED suggests.** From position 3 downward
the belief holds: 0.394% → 0.303% → 0.330% → 0.145% → 0.036%, a ~11x fall with one small blip.
Position genuinely drives CTR across most of the range, so grouping by position is fair there.

**The break is at the very top, and it is sharp.** Pages ranking 1–2 pool to **0.118%** — the
lowest of any shallow band, a third of the 3–5 peak — and **35% of them took zero clicks all
month**, the worst zero-click rate outside position 50+. A page shown thousands of times at
position 1 that nobody clicks is not a normal shape.

**This explains an ML-04 open question.** The `top_3` tier is bands 1–2 and 2–3 pooled:
1.99M impressions at 0.118% plus 9.50M at 0.358% gives 0.316% — matching ML-04's 0.317%, which
sat below `page_1`. Finer banding did not hide the pattern, it located it: one band, not the
whole tier.

### Signal 2 — is a CTR gap trustworthy below the volume floor?

My eligibility line throws away every page under 500 impressions. That is a big claim — it removes
most of the population — so it needs evidence, not a round number that looks tidy. The belief:
**below the floor, CTR is mostly noise**, because a handful of clicks would swing it completely.

This one runs on **all** pages, floor included and excluded, or it could not test the floor.

In [5]:
VB  = [0, 100, 500, 2000, 10000, np.inf]
VL  = ["<100", "100-500", "500-2k", "2k-10k", "10k+"]
pages["vol_band"] = pd.cut(pages.impressions_31d, bins=VB, labels=VL)

grp = pages.groupby("vol_band", observed=True)
s2 = pd.DataFrame({
    "n_pages":         grp.size(),
    "share_zero_click": grp.clicks_31d.apply(lambda s: (s == 0).mean()).round(3),
    "pooled_ctr_pp":   (100 * grp.clicks_31d.sum() / grp.impressions_31d.sum()).round(3),
    "ctr_spread_iqr":  (grp.ctr_pp.quantile(0.75) - grp.ctr_pp.quantile(0.25)).round(3),
    "one_click_worth_pp": (100 / grp.impressions_31d.median()).round(3),
})
print("SIGNAL 2 - how trustworthy is a CTR number, by impression volume?")
print("(one_click_worth_pp = how many percentage points ONE extra click moves the median page)")
display(s2)

lo, hi = s2.iloc[0], s2.iloc[-1]
verdict_2 = ("CONFIRMED" if (lo.share_zero_click > hi.share_zero_click
                             and lo.one_click_worth_pp > 10 * hi.one_click_worth_pp) else
             "MIXED"     if lo.share_zero_click > hi.share_zero_click else "FALSE")
print(f"\nunder 100 impressions: {lo.share_zero_click:.1%} of pages have zero clicks, and one click "
      f"moves the median page by {lo.one_click_worth_pp:.2f}pp")
print(f"at 10k+:               {hi.share_zero_click:.1%} zero-click, one click moves it "
      f"{hi.one_click_worth_pp:.4f}pp")
print(f"\nVERDICT 2: {verdict_2}")

SIGNAL 2 - how trustworthy is a CTR number, by impression volume?
(one_click_worth_pp = how many percentage points ONE extra click moves the median page)


,n_pages,share_zero_click,pooled_ctr_pp,ctr_spread_iqr,one_click_worth_pp
vol_band,,,,,
<100,75506,0.931,0.346,0.000,7.692
100-500,39356,0.682,0.229,0.315,0.441
500-2k,32012,0.293,0.273,0.351,0.104
2k-10k,23987,0.056,0.304,0.339,0.026
10k+,5877,0.010,0.293,0.296,0.006



under 100 impressions: 93.1% of pages have zero clicks, and one click moves the median page by 7.69pp
at 10k+:               1.0% zero-click, one click moves it 0.0060pp

VERDICT 2: CONFIRMED


**Reading verdict 2.** A CTR gap computed on a few dozen impressions is not a small measurement
— it is not a measurement. One click flips it. **What I changed:** the 500-impression floor stays,
and it is now justified by a number rather than by habit; and §2 adds a second gate in *clicks*, not
just in percentage points, so the queue cannot fill up with technically-true but tiny shortfalls.

**One honest note on both verdicts:** these are observed patterns in one month for 36 clients. They
describe this slice; they are not facts about Google.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Three steps, in the session's order: peer baseline → threshold (chosen and documented) → rank by
what is at stake.

**The peer baseline is leave-one-out and volume-weighted**, carried over from ML-04: a page is never
part of the benchmark it is judged against, and the benchmark is pooled clicks ÷ pooled impressions
rather than an average of per-page rates.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- the gap: how far below its position peers does this page sit? ----------
tier_clicks = lane.groupby("position_tier").clicks_31d.transform("sum")
tier_imps   = lane.groupby("position_tier").impressions_31d.transform("sum")

lane["peer_pp"] = (100 * (tier_clicks - lane.clicks_31d) /
                   (tier_imps - lane.impressions_31d)).replace([np.inf, -np.inf], np.nan)
lane["ctr_gap_pp"]             = lane.peer_pp - lane.ctr_pp
lane["expected_missed_clicks"] = lane.impressions_31d * lane.ctr_gap_pp / 100

lane = lane.dropna(subset=["ctr_gap_pp"]).copy()
print(f"scored pages: {len(lane):,}")
print(f"pages sitting below their position peers: {(lane.ctr_gap_pp > 0).sum():,} "
      f"({(lane.ctr_gap_pp > 0).mean():.1%})")

scored pages: 61,881
pages sitting below their position peers: 41,178 (66.5%)


In [7]:
# --- the threshold: strict / balanced / loose, then a documented choice -----
options = []
for label, gap_min, clicks_min in [("loose    (gap > 0, any size)", 0.0, 0),
                                   ("balanced (gap >= 0.10pp, >= 10 clicks)", 0.10, 10),
                                   ("strict   (gap >= 0.25pp, >= 50 clicks)", 0.25, 50)]:
    hit = (lane.ctr_gap_pp >= gap_min) & (lane.expected_missed_clicks >= clicks_min) & (lane.ctr_gap_pp > 0)
    options.append({"cutoff": label, "pages_flagged": int(hit.sum()),
                    "share_of_lane": round(hit.mean(), 3),
                    "clicks_at_stake": round(lane.loc[hit, "expected_missed_clicks"].sum())})
display(pd.DataFrame(options))

GAP_MIN, CLICKS_MIN = 0.10, 10
print(f"\nCHOSEN: gap >= {GAP_MIN}pp AND expected missed clicks >= {CLICKS_MIN}. Why, in writing:")
print(" - the pp floor keeps out shortfalls smaller than the noise between comparable pages;")
print(" - the clicks floor is the session's 'what is at stake' test -- a real gap on a page nobody")
print("   sees is not worth an editor's hour, and signal 2 showed why small pages mislead;")
print(" - loose flags too much of the lane to be a queue; strict leaves real work unflagged.")
print("This is a starting point I expect to move once reviewers tell me what they reject.")

,cutoff,pages_flagged,share_of_lane,clicks_at_stake
0,"loose (gap > 0, any size)",41178,0.665,278665
1,"balanced (gap >= 0.10pp, >= 10 clicks)",6019,0.097,161145
2,"strict (gap >= 0.25pp, >= 50 clicks)",285,0.005,27455



CHOSEN: gap >= 0.1pp AND expected missed clicks >= 10. Why, in writing:
 - the pp floor keeps out shortfalls smaller than the noise between comparable pages;
 - the clicks floor is the session's 'what is at stake' test -- a real gap on a page nobody
   sees is not worth an editor's hour, and signal 2 showed why small pages mislead;
 - loose flags too much of the lane to be a queue; strict leaves real work unflagged.
This is a starting point I expect to move once reviewers tell me what they reject.


In [8]:
# --- score, ONE reason code, action label, rank -----------------------------
lane["score"] = np.where(lane.ctr_gap_pp > 0, lane.expected_missed_clicks, 0.0)

flagged = (lane.ctr_gap_pp >= GAP_MIN) & (lane.expected_missed_clicks >= CLICKS_MIN)
lane["reason_code"] = np.where(flagged, "CTR_BELOW_POSITION_PEERS", "WITHIN_PEER_RANGE")
lane["action"]      = np.where(flagged, "SNIPPET_REVIEW",           "MONITOR")

lane = lane.sort_values("score", ascending=False).reset_index(drop=True)
lane["rank"] = np.arange(1, len(lane) + 1)

print(lane.action.value_counts().to_string())
print(f"\nqueue: {int(flagged.sum()):,} pages for snippet review "
      f"({flagged.mean():.1%} of the eligible lane), carrying "
      f"{lane.loc[lane.action == 'SNIPPET_REVIEW', 'expected_missed_clicks'].sum():,.0f} "
      f"estimated missed clicks per month.")
print(f"the other {int((~flagged).sum()):,} pages are MONITOR -- the queue is allowed to say "
      f"'no action yet', which is what keeps it readable.")

action
MONITOR           55862
SNIPPET_REVIEW     6019

queue: 6,019 pages for snippet review (9.7% of the eligible lane), carrying 161,145 estimated missed clicks per month.
the other 55,862 pages are MONITOR -- the queue is allowed to say 'no action yet', which is what keeps it readable.


In [9]:
# --- write the ranked queue -------------------------------------------------
import os, json
os.makedirs("work/outputs", exist_ok=True)

COLS = ["rank", "client_hash_id", "content_hash_id", "action", "reason_code", "score",
        "impressions_31d", "clicks_31d", "ctr_pp", "peer_pp", "ctr_gap_pp",
        "expected_missed_clicks", "avg_position_31d", "position_tier",
        "days_with_impressions_31d", "position_volatility_31d", "top_day_impression_share",
        "momentum_log14v14"]
queue = lane[COLS].copy()
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"wrote work/outputs/baseline_action_score.csv -- {len(queue):,} rows x {len(COLS)} columns")
print("(the CSV is regenerated on every run and stays out of git by design; the JSON below is the "
      "committed receipt)")
display(queue.head(3))

wrote work/outputs/baseline_action_score.csv -- 61,881 rows x 18 columns
(the CSV is regenerated on every run and stays out of git by design; the JSON below is the committed receipt)


,rank,client_hash_id,content_hash_id,action,reason_code,score,impressions_31d,clicks_31d,ctr_pp,peer_pp,ctr_gap_pp,expected_missed_clicks,avg_position_31d,position_tier,days_with_impressions_31d,position_volatility_31d,top_day_impression_share,momentum_log14v14
0,1,client_23a62021009f63c4,content_44f34c0a90047651,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,660.691234,212404.0,24.0,0.011299,0.322353,0.311054,660.691234,1.665877,top_3,31,4.565167,0.188716,3.460018
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,458.727011,134984.0,1.0,0.000741,0.340579,0.339838,458.727011,3.693038,page_1,31,2.576839,0.124815,0.215576
2,3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,444.079995,143019.0,43.0,0.030066,0.340570,0.310504,444.079995,4.166132,page_1,31,0.830142,0.272712,-0.521154


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The session's habit: before the list ships, a person reads the top of it and asks one question per
row — **"what would make this recommendation wrong?"** A review that finds nothing suspicious was a
shallow review, not a perfect rule.

The caveat for each row below is generated from that row's own numbers, so it names the specific
risk rather than a generic disclaimer. The categories come from the two signal verdicts and from
ML-04's limitations.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def confidence(r):
    if r.impressions_31d >= 10000 and r.days_with_impressions_31d >= 25: return "HIGH"
    if r.impressions_31d >= 2000:                                        return "MEDIUM"
    return "LOW"

def what_would_make_it_wrong(r):
    """Returns (category, sentence) -- the category makes the mix readable in aggregate."""
    if r.clicks_31d == 0:
        return ("zero-click page",
                f"zero clicks on {r.impressions_31d:,.0f} impressions -- a handful of clicks would "
                f"erase most of this gap; the page may also be one nobody ever intends to click")
    if r.top_day_impression_share > 0.30:
        return ("one-day spike",
                f"{r.top_day_impression_share:.0%} of the month's impressions landed on ONE day -- "
                f"a spike, not steady demand, so the monthly CTR describes an unusual month")
    if r.momentum_log14v14 < -0.5:
        return ("demand falling",
                "impressions fell sharply between the two halves of March -- demand may be leaving "
                "this topic, and a better snippet cannot fix falling demand (or a season turning)")
    if r.position_volatility_31d > 10:
        return ("unstable position",
                f"position swung a lot within the month (sd {r.position_volatility_31d:.1f}) -- the "
                f"page was not really in one peer group, so the comparison may be the wrong one")
    if r.avg_position_31d <= 3:
        return ("top-band anomaly",
                "position 1-3, where signal 1 found the break: the 1-2 band pools to 0.118% CTR "
                "with 35% of pages taking zero clicks -- the lowest of any shallow band. Until "
                "that is explained, a large gap here may be measuring the anomaly, not a snippet")
    if r.ctr_gap_pp < 0.20:
        return ("small gap, big page",
                "the shortfall is small in percentage terms and only ranks high because the page is "
                "large -- worth an editor's time only if the rewrite is quick")
    return ("intent may be non-click",
            "the page may serve a query where nobody clicks through (a definition, a price) -- "
            "high impressions with low CTR can be the correct outcome, not a fault")

top = lane.head(20).copy()
top["confidence"] = top.apply(confidence, axis=1)
_caveats = top.apply(what_would_make_it_wrong, axis=1)
top["caveat_category"]           = [c for c, _ in _caveats]
top["what_would_make_it_wrong"]  = [s for _, s in _caveats]

for _, r in top.iterrows():
    print(f"\n#{r['rank']:<3} {r.action:<15} {r.reason_code}   [{r.confidence} confidence]")
    print(f"     why it is here : {r.impressions_31d:,.0f} impressions at position "
          f"{r.avg_position_31d:.1f} ({r.position_tier}); CTR {r.ctr_pp:.2f}% vs peers "
          f"{r.peer_pp:.2f}% -> gap {r.ctr_gap_pp:.2f}pp = ~{r.expected_missed_clicks:,.0f} "
          f"clicks/month at stake")
    print(f"     would be wrong if: {r.what_would_make_it_wrong}")


#1   SNIPPET_REVIEW  CTR_BELOW_POSITION_PEERS   [HIGH confidence]
     why it is here : 212,404 impressions at position 1.7 (top_3); CTR 0.01% vs peers 0.32% -> gap 0.31pp = ~661 clicks/month at stake
     would be wrong if: position 1-3, where signal 1 found the break: the 1-2 band pools to 0.118% CTR with 35% of pages taking zero clicks -- the lowest of any shallow band. Until that is explained, a large gap here may be measuring the anomaly, not a snippet

#2   SNIPPET_REVIEW  CTR_BELOW_POSITION_PEERS   [HIGH confidence]
     why it is here : 134,984 impressions at position 3.7 (page_1); CTR 0.00% vs peers 0.34% -> gap 0.34pp = ~459 clicks/month at stake
     would be wrong if: the page may serve a query where nobody clicks through (a definition, a price) -- high impressions with low CTR can be the correct outcome, not a fault

#3   SNIPPET_REVIEW  CTR_BELOW_POSITION_PEERS   [HIGH confidence]
     why it is here : 143,019 impressions at position 4.2 (page_1); CTR 0.03% vs peers 0.34

In [11]:
# The same review as a table, and what the caveats add up to
display(top[["rank", "action", "confidence", "impressions_31d", "avg_position_31d",
             "ctr_pp", "peer_pp", "ctr_gap_pp", "expected_missed_clicks"]].round(2))

print("caveat mix across the top 20:")
print(top.caveat_category.value_counts().to_string())
print("\nconfidence mix:", {k: int(v) for k, v in top.confidence.value_counts().items()})
print(f"clients represented in the top 20: {top.client_hash_id.nunique()} "
      f"(largest contributes {top.client_hash_id.value_counts().iloc[0]} rows)")

,rank,action,confidence,impressions_31d,avg_position_31d,ctr_pp,peer_pp,ctr_gap_pp,expected_missed_clicks
0,1,SNIPPET_REVIEW,HIGH,212404.0,1.67,0.01,0.32,0.31,660.69
1,2,SNIPPET_REVIEW,HIGH,134984.0,3.69,0.00,0.34,0.34,458.73
2,3,SNIPPET_REVIEW,HIGH,143019.0,4.17,0.03,0.34,0.31,444.08
3,4,SNIPPET_REVIEW,HIGH,203497.0,3.47,0.14,0.34,0.20,404.00
4,5,SNIPPET_REVIEW,HIGH,124075.0,1.31,0.00,0.32,0.32,396.10
5,6,SNIPPET_REVIEW,HIGH,132593.0,6.95,0.06,0.34,0.28,368.51
6,7,SNIPPET_REVIEW,HIGH,107584.0,10.74,0.01,0.33,0.32,340.75
7,8,SNIPPET_REVIEW,HIGH,170808.0,4.40,0.15,0.34,0.19,319.59
8,9,SNIPPET_REVIEW,HIGH,194337.0,5.55,0.19,0.34,0.15,300.69
9,10,SNIPPET_REVIEW,HIGH,89332.0,8.83,0.00,0.34,0.34,300.16


caveat mix across the top 20:
caveat_category
intent may be non-click    9
demand falling             4
top-band anomaly           3
small gap, big page        3
one-day spike              1

confidence mix: {'HIGH': 20}
clients represented in the top 20: 7 (largest contributes 6 rows)


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks — the ones I would pull before this list reaches an editor

Named below by rank, not by page. A queue nobody argues with is a queue nobody read.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

weak = top[(top.confidence == "LOW") | (top.clicks_31d == 0) |
           (top.top_day_impression_share > 0.30) | (top.avg_position_31d <= 3)]
print(f"weak picks inside the top 20: {len(weak)} of 20\n")
for _, r in weak.iterrows():
    print(f"#{r['rank']:<3} {r.impressions_31d:,.0f} impressions, position {r.avg_position_31d:.1f}, "
          f"CTR {r.ctr_pp:.2f}% -> {r.what_would_make_it_wrong[:100]}")

if len(weak) == 0:
    print("none flagged by the automatic checks -- which per the skill means I should look harder, "
          "not that the rule is perfect. Re-read the list by hand before shipping.")
else:
    print(f"\nOBSERVED: {len(weak)}/20 of the top of my own queue carries a named reason it could be "
          f"wrong. That is the expected shape for a hand-written rule, and it is the argument for a "
          f"human confirming every row before an editor spends an hour on it.")

weak picks inside the top 20: 4 of 20

#1   212,404 impressions, position 1.7, CTR 0.01% -> position 1-3, where signal 1 found the break: the 1-2 band pools to 0.118% CTR with 35% of pages tak
#5   124,075 impressions, position 1.3, CTR 0.00% -> position 1-3, where signal 1 found the break: the 1-2 band pools to 0.118% CTR with 35% of pages tak
#14  83,834 impressions, position 1.1, CTR 0.00% -> 35% of the month's impressions landed on ONE day -- a spike, not steady demand, so the monthly CTR d
#20  80,821 impressions, position 2.4, CTR 0.04% -> position 1-3, where signal 1 found the break: the 1-2 band pools to 0.118% CTR with 35% of pages tak

OBSERVED: 4/20 of the top of my own queue carries a named reason it could be wrong. That is the expected shape for a hand-written rule, and it is the argument for a human confirming every row before an editor spends an hour on it.


In [13]:
# --- Leakage check: nothing future-dated, nothing derived from the answer ---
SCORE_INPUTS = ["impressions_31d", "ctr_gap_pp"]          # everything the score is built from
LABEL_SIDE   = {"clicks_31d", "ctr_pp", "peer_pp", "ctr_gap_pp", "expected_missed_clicks", "score"}
PRODUCT_FLAGS = {"health_score", "needs_ctr_fix", "is_quick_win", "trend_direction", "trend_pct"}

checks = {
    "window ends before the decision moment":
        WINDOW_END < DECISION_MOMENT,
    "no FlyRank product flag is present in the frame":
        len(PRODUCT_FLAGS & set(lane.columns)) == 0,
    "no future partition was read (only month=2026-03)":
        MONTH == "2026-03",
    "eligibility gate applied before the threshold, not after":
        len(lane) <= len(pages),
}
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values())

print("\nOn the one that is NOT a clean pass -- said plainly rather than hidden:")
print("  the score IS built from clicks, because the gap is arithmetic on CTR. That is fine for a")
print("  RULE: a rule describes what already happened and needs no held-out honesty. It would be")
print("  fatal for a MODEL, which is exactly the leak ML-04 demonstrated with clicks_31d. The")
print("  Week-5 model must beat this baseline WITHOUT using clicks as an input.")

PASS  window ends before the decision moment
PASS  no FlyRank product flag is present in the frame
PASS  no future partition was read (only month=2026-03)
PASS  eligibility gate applied before the threshold, not after

On the one that is NOT a clean pass -- said plainly rather than hidden:
  the score IS built from clicks, because the gap is arithmetic on CTR. That is fine for a
  RULE: a rule describes what already happened and needs no held-out honesty. It would be
  fatal for a MODEL, which is exactly the leak ML-04 demonstrated with clicks_31d. The
  Week-5 model must beat this baseline WITHOUT using clicks as an input.


In [14]:
# --- Receipts (this JSON is the committed artefact; the CSV is not) ---------
receipts = {
    "assignment": "ML-07 - Baseline Action Score",
    "lane": "Lane 4 - CTR / engagement opportunity scoring (CONFIRMED, not switched)",
    "slice": {"month": MONTH, "window": [WINDOW_START, WINDOW_END],
              "decision_moment": DECISION_MOMENT,
              "pages_with_gsc": int(len(pages)), "eligible_pages": int(len(lane)),
              "clients": int(lane.client_hash_id.nunique())},
    "eligibility": {"min_impressions": MIN_IMPRESSIONS, "min_active_days": MIN_ACTIVE_DAYS,
                    "min_position": MIN_POSITION,
                    "cooldown": "NOT IMPLEMENTABLE - release has no optimisation history"},
    "signal_checks": {
        "signal_1_ctr_falls_with_position": {"verdict": verdict_1, "flag_linked": True,
                                             "buckets": int(len(s1)), "floor_n": FLOOR},
        "signal_2_volume_makes_ctr_trustworthy": {"verdict": verdict_2, "flag_linked": False,
                                                  "buckets": int(len(s2))},
    },
    "rule": {"threshold_gap_pp": GAP_MIN, "threshold_missed_clicks": CLICKS_MIN,
             "reason_codes": ["CTR_BELOW_POSITION_PEERS", "WITHIN_PEER_RANGE"],
             "actions": ["SNIPPET_REVIEW", "MONITOR"],
             "score": "expected_missed_clicks = impressions * ctr_gap_pp / 100"},
    "queue": {"flagged": int((lane.action == "SNIPPET_REVIEW").sum()),
              "monitor": int((lane.action == "MONITOR").sum()),
              "clicks_at_stake_flagged": round(float(
                  lane.loc[lane.action == "SNIPPET_REVIEW", "expected_missed_clicks"].sum())),
              "clicks_at_stake_top20": round(float(top.expected_missed_clicks.sum()))},
    "top20_review": {"weak_picks": int(len(weak)),
                     "confidence_mix": {k: int(v) for k, v in top.confidence.value_counts().items()},
                     "caveat_mix": {k: int(v) for k, v in top.caveat_category.value_counts().items()}},
    "open_questions": [
        "position 1-2 band pools to 0.118% CTR with 35% zero-click pages -- the lowest of any "
        "shallow band, and the arithmetic source of ML-04's top_3-below-page_1 result",
        "CTR level ~0.3% across the lane vs ~2.78% documented (carried from ML-04, still open)",
    ],
    "proxy_note": "ctr_gap_pp is an AUTHORED proxy, not an observed outcome. No causal claim.",
}
with open("work/outputs/ml07_baseline_receipts.json", "w") as f:
    json.dump(receipts, f, indent=2)

print(f"signals: {verdict_1} / {verdict_2}")
print(f"queue: {receipts['queue']['flagged']:,} SNIPPET_REVIEW | "
      f"{receipts['queue']['monitor']:,} MONITOR | "
      f"{receipts['queue']['clicks_at_stake_flagged']:,} clicks at stake")
print(f"top 20 carries {receipts['queue']['clicks_at_stake_top20']:,} of those, with "
      f"{len(weak)} weak picks named")
print("\nsaved -> work/outputs/ml07_baseline_receipts.json")

signals: MIXED / CONFIRMED
queue: 6,019 SNIPPET_REVIEW | 55,862 MONITOR | 161,145 clicks at stake
top 20 carries 6,478 of those, with 4 weak picks named

saved -> work/outputs/ml07_baseline_receipts.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### What this baseline commits me to

1. **The Week-5 model must beat this rule** on the same slice and the same metric — and it must do
   it *without* `clicks_31d`, which this rule is allowed to use and a model is not.
2. **The baseline is frozen now.** Moving it later to make a model look good convinces nobody.
3. Two things this rule cannot do, both inherited: no cooldown (no optimisation history in the
   release), and no way to tell a real snippet problem from a query where nobody ever clicks.